# EcoHome Energy Advisor - Agent Run & Evaluation

In this notebook, you'll run the Energy Advisor agent with various real-world scenarios and see how it helps customers optimize their energy usage.

## Learning Objectives
- Create the agent's instructions
- Run the Energy Advisor with different types of questions
- Evaluate response quality and accuracy
- Measure tool usage effectiveness
- Identify areas for improvement
- Implement evaluation metrics

## Evaluation Criteria
- **Accuracy**: Correct information and calculations
- **Relevance**: Responses address the user's question
- **Completeness**: Comprehensive answers with actionable advice
- **Tool Usage**: Appropriate use of available tools
- **Reasoning**: Clear explanation of recommendations


## 1. Import and Initialize

In [1]:
from datetime import datetime
from agent import Agent

from deepeval.test_case import LLMTestCase,ToolCall,LLMTestCaseParams
from deepeval.metrics import(
    ContextualRecallMetric,
    ContextualPrecisionMetric,
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    ToolCorrectnessMetric
)
from deepeval import evaluate
from deepeval.models import GPTModel
from deepeval.metrics import GEval

from langchain_core.messages import ToolMessage, AIMessage

from dotenv import load_dotenv
import httpx
import os
from pprint import pprint

In [2]:
load_dotenv()

True

In [3]:
## TODO: Create the agent's instructions

ECOHOME_SYSTEM_PROMPT = """
    You are EcoHome Energy Advisor, helping customers with solar, EVs, and smart thermostats cut electricity cost and carbon footprint.
    Capabilities:
    1. Analyse usage and solar data
    2. Look up weather forecasts
    3. Look up energy prices
    4. Provide tips on energy tips
    5. Calculate savings based in device usage and energy prices

    When a user asks a question, you must:
    1. Determine the question's intent and review any additional context on time and location
    2. Decide which tool to use (if any)
    3. Retrieve relevant data using tools
    4. Reason about how to best answer given the question, context and tool responses (include your reasoning in <think></think> XML tags)
    5. Construct a clear, informative and actionable response, grounded in tool data

    Example:
    <question>When should I charge my electric car tomorrow to minimize cost and maximize solar power?</question>
    <context>Kitchener, Ontario, 2026-11-07 17:47</context>
    <think>I need to see what the upcoming forecast says about weather tomorrow and get energy prices accordingly.</think>
    <tools>['get_weather_forecast','get_electricity_prices']</tools>
    <answer>Set your thermostat to around 20°C (68°F) on Wednesday afternoon, particularly between 3 PM and 7 PM when electricity prices peak at $0.2252 to $0.2489 per kWh. 
    By maintaining a comfortable temperature, you can avoid excessive energy use during these high-cost hours. If you can, consider pre-cooling your home during the off-peak hours (before 6 AM or after 8 PM) when rates are significantly lower.
    </answer>

    Rules:
    - Always search for applicable energy tips before providing a final recommendation.
    - Query for usage and solar data before asking the user for it. 
    - Ground every answer in tool data. Never invent prices, weather, usage, or generation figures.
    - Use only the tools the question needs. If a needed input (location, date, device) is missing from the context, ask once or state your assumption.
    - Recommend specific times and settings. Favor off-peak rates and high solar output.
    - Quantify savings in kWh and USD where possible. State the basis for each number.
    - If a tool errors or returns no data, say so. Don't guess.
    - Keep the answer short and actionable: recommendation first, then a brief rationale.
"""


In [4]:
ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)

In [5]:
response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context=f"Today: {datetime.now()}. Location: San Francisco, CA"
)

In [6]:
print(response["messages"][-1].content)

To minimize costs and maximize solar power when charging your electric car tomorrow (November 9), I recommend charging between 12 PM and 2 PM. 

During this time, the solar output will be at its peak, with significant sunlight available, and the electricity rates are relatively lower compared to the evening peak hours. 

Here's a breakdown:
- **Solar Generation**: Expect high solar irradiance around noon, which means more solar energy will be available.
- **Electricity Prices**: The rates from 12 PM to 2 PM are approximately $0.1679 and $0.1933 per kWh, respectively, which are lower than the peak rates in the evening that can go up to $0.2737 per kWh.

By charging during this window, you can take advantage of both solar energy and lower electricity costs, significantly reducing your overall charging expenses.


In [7]:
print("TOOLS:")
for msg in response["messages"]:
    obj = msg.model_dump()
    if obj.get("tool_call_id"):
        print("-", msg.name)

TOOLS:
- get_weather_forecast
- get_electricity_prices


## 2. Define Test Cases

In [8]:
# TODO: Define comprehensive test cases for the Energy Advisor
# Create 10 test cases covering different scenarios:
# - EV charging optimization
# - Thermostat settings
# - Appliance scheduling
# - Solar power maximization
# - Cost savings calculations

In [9]:
test_cases = [
    {
        "id": "ev_charging_1",
        "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "The response should contain time recommendation, cost analysis and solar consideration",
    },
    {
        "id": "thermostat_1",
        "question": "What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?",
        "expected_tools": ["get_electricity_prices", "get_weather_forecast", "search_energy_tips"],
        "expected_response": "The response should recommend a specific thermostat setting, tie it to peak-period pricing, and consider outdoor temperature (e.g. pre-cooling or pre-heating before the spike and setting back during it)",
    },
    {
        "id": "appliance_1",
        "question": "How much can I save by running my dishwasher during off-peak hours instead of in the evening?",
        "expected_tools": ["get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "The response should compare peak and off-peak rates, recommend a specific off-peak start time, and quantify savings in kWh and USD with the basis for the numbers stated",
    },
    {
        "id": "solar_1",
        "question": "Based on this week's forecast, which days will my solar panels produce the most, and how should I shift my usage to take advantage?",
        "expected_tools": ["get_weather_forecast", "query_solar_generation", "query_energy_usage"],
        "expected_response": "The response should identify the best solar days using forecast and past generation data, and recommend shifting flexible loads (EV, laundry, dishwasher) to midday hours on those days",
    },
    {
        "id": "savings_1",
        "question": "Suggest three ways I can reduce energy use based on my usage history, and estimate the yearly savings.",
        "expected_tools": ["get_recent_energy_summary", "query_energy_usage", "search_energy_tips", "calculate_energy_savings"],
        "expected_response": "The response should give three specific recommendations grounded in the user's device usage data, supported by retrieved tips, with estimated annual savings in USD for each",
    },
    {
        "id":"ev_charging_2",
        "question": "What are three ways I can optimise on the distance my Tesla delivers per charge?",
        "expected_tools": ["search_energy_tips"],
        "expected_response": "The response should be a polite decline, apology or a redirection to a supported query."
    },
    {
        "id": "thermostat_2",
        "question": "Is it worth pre-cooling my house in the morning before the afternoon peak pricing starts?",
        "expected_tools": ["get_electricity_prices", "get_weather_forecast", "get_recent_energy_summary", "search_energy_tips"],
        "expected_response": "The response should give a clear yes or no, explain the trade-off between off-peak and peak rates, suggest a target temperature and timing, and ground the reasoning in the user's HVAC usage",
    },
    {
        "id": "appliance_2",
        "question": "What's the best time to run my washing machine and dryer this weekend?",
        "expected_tools": ["get_electricity_prices", "get_weather_forecast", "search_energy_tips"],
        "expected_response": "The response should recommend specific times for each appliance, favor off-peak or high-solar hours, and avoid peak periods, giving a brief rationale",
    },
    {
        "id": "solar_2",
        "question": "How much of my energy use did my solar panels cover over the past week, and what could I do to use more of it directly?",
        "expected_tools": ["query_solar_generation", "query_energy_usage", "get_recent_energy_summary"],
        "expected_response": "The response should compare total solar generation with total consumption in kWh, state the approximate share covered, and suggest shifting specific loads to peak solar hours",
    },
    {
        "id": "savings_2",
        "question": "If I cut my HVAC usage by 20%, how much would I save per month and per year?",
        "expected_tools": ["query_energy_usage", "get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "The response should base the calculation on actual HVAC consumption from the usage data, show current versus reduced kWh, and give monthly and annual savings in USD with the price assumption stated",
    },
]

if len(test_cases) < 10:
    raise ValueError("You MUST have at least 10 test cases")

## 3. Run Agent Tests

In [10]:

with httpx.Client() as client:
    response=client.get(f"{os.environ["IPINFO_API_URL"]}")

CONTEXT = f"Today: {datetime.now()}. Location: {response.json()["city"]}, {response.json()["regionName"]}"

In [11]:
# Run the agent tests
# For each test case, call the agent and collect the response
# Store results for evaluation

print("=== Running Agent Tests ===")
test_results = []

for i, test_case in enumerate(test_cases):
    print(f"\nTest {i+1}: {test_case['id']}")
    print(f"Question: {test_case['question']}")
    print("-" * 50)
    
    try:
        # Call the agent
        response = ecohome_agent.invoke(
            question=test_case["question"],
            context=CONTEXT
        )
        
        # Store the result
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': response,
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat()
        }
        test_results.append(result)
                
    except Exception as e:
        print(f"Error: {e}")
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': f"Error: {str(e)}",
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat(),
            'error': str(e)
        }
        test_results.append(result)

print(f"\nCompleted {len(test_results)} tests")


=== Running Agent Tests ===

Test 1: ev_charging_1
Question: When should I charge my electric car tomorrow to minimize cost and maximize solar power?
--------------------------------------------------

Test 2: thermostat_1
Question: What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?
--------------------------------------------------

Test 3: appliance_1
Question: How much can I save by running my dishwasher during off-peak hours instead of in the evening?
--------------------------------------------------

Test 4: solar_1
Question: Based on this week's forecast, which days will my solar panels produce the most, and how should I shift my usage to take advantage?
--------------------------------------------------

Test 5: savings_1
Question: Suggest three ways I can reduce energy use based on my usage history, and estimate the yearly savings.
--------------------------------------------------

Test 6: ev_charging_2
Question: What are three way

In [12]:
test_results

[{'test_id': 'ev_charging_1',
  'question': 'When should I charge my electric car tomorrow to minimize cost and maximize solar power?',
  'response': {'messages': [SystemMessage(content='Today: 2026-11-08 19:38:18.649371. Location: Kitchener, Ontario', additional_kwargs={}, response_metadata={}, id='5e9631de-43c9-4028-b79f-a5ef60c572b7'),
    HumanMessage(content='When should I charge my electric car tomorrow to minimize cost and maximize solar power?', additional_kwargs={}, response_metadata={}, id='85b8887d-9941-4f54-9689-b5dd3c1b92c7'),
    AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_3p1lFsOgN7RZ2jJKM6AcVllA', 'function': {'arguments': '{"location": "Kitchener, Ontario", "days": 1}', 'name': 'get_weather_forecast'}, 'type': 'function'}, {'id': 'call_6JlpX3hD4IR0rQtZhTk04PJb', 'function': {'arguments': '{"date": "2026-11-09"}', 'name': 'get_electricity_prices'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 62

## 4. Evaluate Responses

In [13]:
# TODO: Implement evaluation functions
# Create functions to evaluate:
# - Final Response
# - Tool usage


llm_judge = GPTModel(
    model="gpt-4o-mini",
    temperature=0.0,
    base_url=os.environ["OPENAI_BASE_URL"],
    api_key=os.environ["VOCAREUM_API_KEY"]
)

class CompletenessMetric(GEval):
    def __init__(self, model=None,threshold=0.5,**kwargs):
        super().__init__(
            name = "Completeness",
            criteria=("How well does the answer compare against the expected response?"),
            evaluation_params = [LLMTestCaseParams.INPUT,
                                LLMTestCaseParams.ACTUAL_OUTPUT,
                                LLMTestCaseParams.EXPECTED_OUTPUT],
            model=model,
            threshold=threshold,
            **kwargs

        )

class UsefulnessMetric(GEval):
    def __init__(self, model=None, threshold=0.5, **kwargs):
        super().__init__(
            name="Usefulness",
            criteria=("Does the answer include concrete, actionable steps, specific times or settings, and clear next steps?"),
            evaluation_params = [LLMTestCaseParams.INPUT,
                                LLMTestCaseParams.ACTUAL_OUTPUT,
                                LLMTestCaseParams.EXPECTED_OUTPUT],
            model=model,
            threshold=threshold,
            **kwargs
        
        )
    

class AccuracyMetric(GEval):
    def __init__(self,model=None,threshold=0.5,**kwargs):
        super().__init__(
            name="Accuracy",
            criteria=("Does the answer leverage correct reasoning and calculations to arrive at appropriate answers and forecasts?"),
            evaluation_params = [LLMTestCaseParams.INPUT,
                                                    LLMTestCaseParams.ACTUAL_OUTPUT,
                                                    LLMTestCaseParams.EXPECTED_OUTPUT],
            model=model,
            threshold=threshold,
            **kwargs

        )
    

metrics = {

    "relevance":AnswerRelevancyMetric(threshold=0.5, model=llm_judge),
    #"tool_correctness":ToolCorrectnessMetric(threshold=0.5, model=llm_judge),
    "completeness":CompletenessMetric(threshold=0.5, model=llm_judge),
    "accuracy":AccuracyMetric(threshold=0.5, model=llm_judge),
    "usefulness":UsefulnessMetric(threshold=0.5, model=llm_judge)

}

In [14]:

def extract_tools_called(messages:list) -> list[ToolCall]:
    # Get messages from response
    # messages = response["messages"]
    # Isolate the ToolMessages which are in response to AIMessages with tool calls, keyed by ID
    tool_messages = {m.tool_call_id : m.content for m in messages if isinstance(m, ToolMessage)}
    # Dump each tool call and its arguments with outputs into a list
    calls = []
    for m in messages:
        if isinstance(m, AIMessage):
            for call in m.tool_calls:
                calls.append(ToolCall(
                    name=call["name"],
                    input_parameters=call["args"],
                    output=tool_messages.get(call["id"])
                ))
            
    
    # Return the list
    return calls


In [15]:
# TODO: Create a response evaluator
def evaluate_response(results:dict,metrics:dict) -> dict:
    """Evaluate a single response against expected response"""
    scores = {}
    for name,metric in metrics.items():
        metric.measure(LLMTestCase(
                                        input=results["question"],
                                        actual_output=results["response"]["messages"][-1].content,
                                        expected_output=results["expected_response"],
                                        tools_called=extract_tools_called(results["response"]["messages"]),
                                        expected_tools=[ToolCall(name=n) for n in results["expected_tools"]]
                                        )
                                    )
        scores[name] = {"score":metric.score,
                        "reason":metric.reason}
        

    return scores



In [19]:
# TODO: Create a tool usage evaluator
def evaluate_tool_usage(messages: list, expected_tools: list):
    """Compare tools called by the agent vs expected tools."""
    tools_called = extract_tools_called(messages)
    called_tool_set = {call.name for call in tools_called}
    expected_tool_set = set(expected_tools)

    correct,missing,unnecessary = called_tool_set & expected_tool_set, expected_tool_set - called_tool_set, called_tool_set - expected_tool_set

    return{
        "completeness_score":len(correct)/len(expected_tool_set) if expected_tool_set else 0,
        "appropriateness_score":len(correct)/len(called_tool_set) if called_tool_set else 0,
        "correct_tools": sorted(correct),
        "missing_tools": sorted(missing),
        "unnecessary_tools": sorted(unnecessary)
    }

In [20]:
# TODO: Generate a comprehensive evaluation report
# Calculate overall scores and metrics
# Identify strengths and weaknesses
# Provide recommendations for improvement
def generate_evaluation_report(results: list) -> dict:
    final_mapped_scores = {}
    for test_result in results:
        final_mapped_scores[test_result["test_id"]] = {"response": evaluate_response(test_result, metrics),
                                                        "tool_usage": evaluate_tool_usage(test_result["response"]["messages"],test_result["expected_tools"])}
    
    return final_mapped_scores
    

In [21]:
report = generate_evaluation_report(test_results)
pprint(report,width=100,sort_dicts=False)

Output()

Output()

Output()

Output()

Output()

Output()

... (31 outputs left out)

Output()

Output()

Output()

{'ev_charging_1': {'response': {'relevance': {'score': 1.0,
                                              'reason': 'The score is 1.00 because the response '
                                                        'directly addresses the question about '
                                                        'charging the electric car to minimize '
                                                        'cost and maximize solar power without any '
                                                        'irrelevant statements.'},
                                'completeness': {'score': 0.9223365505743051,
                                                 'reason': 'The Actual Output effectively '
                                                           'addresses the Input by providing a '
                                                           'specific time recommendation (12 PM to '
                                                           '2 PM) that aligns with both cost '
 

In [26]:
def calculate_statistics(per_test_results: dict, threshold: float = 0.5):
    by_metric = {}
    for t in per_test_results.values():
        for name, v in t["response"].items():
            by_metric.setdefault(name, []).append(v["score"])
        for name in ("completeness_score", "appropriateness_score"):
            by_metric.setdefault(f"tool_{name}", []).append(t["tool_usage"][name])

    overall = {name: round(sum(s) / len(s), 2) for name, s in by_metric.items()}
    overall["overall_mean"] = round(sum(overall.values()) / len(overall), 2)

    metric_means = {k: v for k, v in overall.items() if k != "overall_mean"}
    strengths = sorted([k for k, v in metric_means.items() if v >= threshold], key=lambda k: -metric_means[k])
    weaknesses = sorted([k for k, v in metric_means.items() if v < threshold], key=lambda k: metric_means[k])

    # Most frequently missing / unnecessary tools across all tests
    missing, extra = {}, {}
    for t in per_test_results.values():
        for tool in t["tool_usage"]["missing_tools"]:
            missing[tool] = missing.get(tool, 0) + 1
        for tool in t["tool_usage"]["unnecessary_tools"]:
            extra[tool] = extra.get(tool, 0) + 1

    recommendations = [f"Improve '{w}' (mean {metric_means[w]}): review the lowest-scoring tests and their reasons." for w in weaknesses]
    if missing:
        top = max(missing, key=missing.get)
        recommendations.append(f"'{top}' was missed in {missing[top]} test(s): strengthen the prompt rule that requires it.")
    if extra:
        top = max(extra, key=extra.get)
        recommendations.append(f"'{top}' was called unnecessarily in {extra[top]} test(s): tighten when it should be used.")
    if not recommendations:
        recommendations.append("All metrics meet the threshold. Add harder test cases to find remaining gaps.")

    return {
        "per_test": per_test_results,
        "overall": overall,
        "strengths": strengths,
        "weaknesses": weaknesses,
        "recommendations": recommendations,
    }

In [27]:
# Cell 22: display only
def display_evaluation_report(report: dict, show_reasons: bool = False) -> None:
    line = "=" * 60
    print(f"{line}\nENERGY ADVISOR EVALUATION REPORT\n{line}")

    print("\nOVERALL SCORES")
    for name, score in report["overall"].items():
        print(f"  {name:<28} {score:.2f}")

    print("\nSTRENGTHS")
    for s in report["strengths"] or ["(none)"]:
        print(f"  + {s}")
    print("\nWEAKNESSES")
    for w in report["weaknesses"] or ["(none)"]:
        print(f"  - {w}")

    print("\nRECOMMENDATIONS")
    for i, rec in enumerate(report["recommendations"], 1):
        print(f"  {i}. {rec}")

    print(f"\n{line}\nPER-TEST DETAIL\n{line}")
    for test_id, t in report["per_test"].items():
        scores = "  ".join(f"{k}={v['score']:.2f}" for k, v in t["response"].items())
        tu = t["tool_usage"]
        print(f"\n[{test_id}]  {scores}")
        print(f"  tools: completeness={tu['completeness_score']:.2f}  appropriateness={tu['appropriateness_score']:.2f}")
        print(f"    correct:     {tu['correct_tools'] or '-'}")
        print(f"    missing:     {tu['missing_tools'] or '-'}")
        print(f"    unnecessary: {tu['unnecessary_tools'] or '-'}")
        if show_reasons:
            for k, v in t["response"].items():
                print(f"    {k} reason: {v['reason']}")


In [28]:
display_evaluation_report(calculate_statistics(report))

ENERGY ADVISOR EVALUATION REPORT

OVERALL SCORES
  relevance                    0.83
  completeness                 0.74
  accuracy                     0.70
  usefulness                   0.76
  tool_completeness_score      0.74
  tool_appropriateness_score   0.92
  overall_mean                 0.78

STRENGTHS
  + tool_appropriateness_score
  + relevance
  + usefulness
  + completeness
  + tool_completeness_score
  + accuracy

WEAKNESSES
  - (none)

RECOMMENDATIONS
  1. 'search_energy_tips' was missed in 3 test(s): strengthen the prompt rule that requires it.
  2. 'query_energy_usage' was called unnecessarily in 1 test(s): tighten when it should be used.

PER-TEST DETAIL

[ev_charging_1]  relevance=1.00  completeness=0.92  accuracy=0.87  usefulness=0.89
  tools: completeness=1.00  appropriateness=1.00
    correct:     ['get_electricity_prices', 'get_weather_forecast']
    missing:     -
    unnecessary: -

[thermostat_1]  relevance=0.70  completeness=0.91  accuracy=0.87  usefulness=0.9